# Direct multi-layout invoice result (no training)
This notebook sends **one complete PDF** to the Claude API and returns the app's full bilingual invoice JSON with arithmetic checks. No verified labels, GPU, Google Drive, local OCR, or model training are needed. Choose a PDF already in the public repo or upload a new one.

Before cell 3, create an API key in the [Claude Console](https://platform.claude.com/settings/keys) and save it in Colab's left-side **Secrets** as `ANTHROPIC_API_KEY`; turn **Notebook access** on. A Claude chat subscription does not include API usage. **Each run sends one PDF to Anthropic and may incur API charges.** The key is never printed or stored in GitHub. The result is not independently source-verified; `needs_review` and arithmetic checks remain visible.

In [ ]:
#@title 1. Load the current public OCR code
import pathlib, subprocess, sys
PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ubaid-148/OCR.git', str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    raise ValueError('/content/OCR is not a Git clone; use a fresh runtime')
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
print('Ready:', PROJECT_DIR)

In [ ]:
#@title 2. Select ONE PDF
PDF_SOURCE = 'repo' #@param ['repo', 'upload']
PDF_NAME = '9498.pdf' #@param {type:'string'}
if PDF_SOURCE == 'repo':
    if pathlib.Path(PDF_NAME).name != PDF_NAME or not PDF_NAME.lower().endswith('.pdf'):
        raise ValueError('PDF_NAME must be one PDF filename, such as 9498.pdf')
    selected_pdf = PROJECT_DIR / 'public_invoice_pdfs' / PDF_NAME
    if not selected_pdf.is_file(): raise FileNotFoundError(selected_pdf)
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Upload exactly one PDF')
    name, content = next(iter(uploaded.items()))
    if not name.lower().endswith('.pdf'): raise ValueError('Upload a PDF file')
    selected_pdf = pathlib.Path('/content') / pathlib.Path(name).name
    selected_pdf.write_bytes(content)
print('Selected:', selected_pdf.name, 'bytes:', selected_pdf.stat().st_size)

In [ ]:
#@title 3. Read the complete PDF with Claude (ONE paid API request)
MODEL_ID = 'claude-sonnet-5' #@param {type:'string'}
MAX_OUTPUT_TOKENS = 16384 #@param {type:'integer'}
from google.colab import userdata
try:
    api_key = userdata.get('ANTHROPIC_API_KEY')
except Exception as error:
    raise RuntimeError('Add ANTHROPIC_API_KEY in Colab Secrets and enable Notebook access. Do not paste the key into a cell.') from error
if not api_key: raise ValueError('ANTHROPIC_API_KEY is empty in Colab Secrets')
from cloud_invoice import extract_invoice_claude
result = extract_invoice_claude(selected_pdf, api_key, model=MODEL_ID, max_tokens=MAX_OUTPUT_TOKENS)
import json
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
#@title 4. Save and download the full result
DOWNLOAD_RESULT = True #@param {type:'boolean'}
output_path = pathlib.Path('/content') / (selected_pdf.stem + '-invoice-result.json')
output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', output_path)
if DOWNLOAD_RESULT:
    from google.colab import files
    files.download(str(output_path))